In [ ]:
# CELL 1: Check installation
import sys
print(f"Python version: {sys.version}")
print("\nChecking packages...")

# Check numpy
import numpy as np
print(f"✓ numpy version: {np.__version__}")

# Check gymnasium
import gymnasium as gym
print(f"✓ gymnasium version: {gym.__version__}")

# Check ale_py
import ale_py
print(f"✓ ale_py version: {ale_py.__version__}")

# Check stable-baselines3
import stable_baselines3 as sb3
print(f"✓ stable-baselines3 version: {sb3.__version__}")

# Check GPU
import torch
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected. Please enable T4 GPU:")
    print("   Runtime → Change runtime type → T4 GPU")

print("\n✅ All packages installed correctly!")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Checking packages...
✓ numpy version: 1.26.4
✓ gymnasium version: 1.3.0
✓ ale_py version: 0.10.0
✓ stable-baselines3 version: 2.9.0
✓ PyTorch version: 2.11.0+cu128
✓ GPU Available: True
✓ GPU Name: Tesla T4

✅ All packages installed correctly!


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# CELL 3: Test Atari Environment
import gymnasium as gym
import ale_py

# Register Atari environments
gym.register_envs(ale_py)

print("Testing Atari environment...")
try:
    env = gym.make("ALE/Pong-v5")
    obs, _ = env.reset()
    print(f"✅ SUCCESS!")
    print(f"   Environment: ALE/Pong-v5")
    print(f"   Observation shape: {obs.shape}")
    print(f"   Action space: {env.action_space}")
    env.close()
except Exception as e:
    print(f"❌ Error: {e}")

print("\n✅ Environment is ready for training!")

Testing Atari environment...
✅ SUCCESS!
   Environment: ALE/Pong-v5
   Observation shape: (210, 160, 3)
   Action space: Discrete(6)

✅ Environment is ready for training!


In [ ]:
# CELL 4: Train DQN on Pong (1 Million Steps)
import os
import time
import numpy as np
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)

from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy

# ================================================================
# ATARI WRAPPERS
# ================================================================

class MaxAndSkipEnv(gym.Wrapper):
    def __init__(self, env, skip=4):
        super().__init__(env)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        done = False
        truncated = False
        for _ in range(self._skip):
            obs, reward, done, truncated, info = self.env.step(action)
            total_reward += reward
            if done or truncated:
                break
        return obs, total_reward, done, truncated, info

class WarpFrame(gym.ObservationWrapper):
    def __init__(self, env, width=84, height=84):
        super().__init__(env)
        self.width = width
        self.height = height
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=(height, width, 1), dtype=np.uint8
        )

    def observation(self, frame):
        import cv2
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        frame = cv2.resize(frame, (self.width, self.height), interpolation=cv2.INTER_AREA)
        return np.expand_dims(frame, -1)

class ClipRewardEnv(gym.RewardWrapper):
    def __init__(self, env):
        super().__init__(env)

    def reward(self, reward):
        return np.sign(reward)

class FrameStack(gym.Wrapper):
    def __init__(self, env, n_frames=4):
        super().__init__(env)
        self.n_frames = n_frames
        self.frames = []
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=(n_frames, 84, 84), dtype=np.uint8
        )

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.frames = [obs] * self.n_frames
        return self._get_obs(), info

    def step(self, action):
        obs, reward, done, truncated, info = self.env.step(action)
        self.frames.append(obs)
        self.frames.pop(0)
        return self._get_obs(), reward, done, truncated, info

    def _get_obs(self):
        return np.concatenate(self.frames, axis=2).transpose(2, 0, 1)

def create_full_env():
    env = gym.make("ALE/Pong-v5")
    env = MaxAndSkipEnv(env, skip=4)
    env = WarpFrame(env, width=84, height=84)
    env = ClipRewardEnv(env)
    env = FrameStack(env, n_frames=4)
    return env

# ================================================================
# TRAINING CALLBACK
# ================================================================

class TrainingLogger(BaseCallback):
    def __init__(self, verbose=1):
        super().__init__(verbose)
        self.episode_rewards = []
        self.episode_count = 0

    def _on_step(self):
        if 'ep_info_buffer' in self.locals and len(self.locals['ep_info_buffer']) > self.episode_count:
            ep_info = self.locals['ep_info_buffer'][-1]
            self.episode_count += 1
            self.episode_rewards.append(ep_info['r'])

            if self.verbose > 0 and self.episode_count % 50 == 0:
                recent_mean = np.mean(self.episode_rewards[-50:]) if len(self.episode_rewards) >= 50 else 0
                print(f"Episode {self.episode_count}: Reward={ep_info['r']:.2f}, Recent 50 Mean={recent_mean:.2f}")
        return True

# ================================================================
# START TRAINING
# ================================================================

print("\n" + "="*70)
print("TRAINING DQN ON PONG - 1,000,000 TIMESTEPS")
print("="*70)

# Check GPU
import torch
if torch.cuda.is_available():
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: No GPU detected! Training will be slow.")
    print("   Go to: Runtime → Change runtime type → T4 GPU")
    print("   Then restart runtime and run this cell again.")
    print("="*70 + "\n")

print("Estimated time: 45-60 minutes on T4 GPU")
print("="*70 + "\n")

# Create environment
env = DummyVecEnv([create_full_env])

# Create model
model = DQN(
    'CnnPolicy',
    env,
    learning_rate=0.0001,
    gamma=0.99,
    batch_size=32,
    buffer_size=100000,
    exploration_fraction=0.1,
    exploration_final_eps=0.01,
    exploration_initial_eps=1.0,
    target_update_interval=1000,
    train_freq=4,
    gradient_steps=1,
    learning_starts=10000,
    verbose=1,
    tensorboard_log='./tensorboard_logs/'
)

# Train
start_time = time.time()
callback = TrainingLogger(verbose=1)

try:
    model.learn(
        total_timesteps=1000000,
        callback=callback,
        log_interval=100
    )
except KeyboardInterrupt:
    print("\n⚠️ Training interrupted. Saving model...")

training_time = time.time() - start_time

# Save model
model.save("dqn_model")
print(f"\n✅ Model saved!")
print(f"   Training time: {training_time/60:.2f} minutes")
print(f"   Total episodes: {len(callback.episode_rewards)}")

# Evaluate
print("\n" + "="*70)
print("EVALUATING TRAINED MODEL")
print("="*70)

eval_env = create_full_env()
mean_reward, std_reward = evaluate_policy(
    model, eval_env, n_eval_episodes=10, deterministic=True
)
print(f"Mean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"Random baseline: -21.00")
print(f"Improvement: {mean_reward - (-21.00):.2f}")

# Download model
from google.colab import files
files.download("dqn_model.zip")

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)
print("The model has been downloaded as dqn_model.zip")
print("Place it in your D:/formative3_group13/ folder")
print("Run: python play.py")


TRAINING DQN ON PONG - 1,000,000 TIMESTEPS
✅ Using GPU: Tesla T4
Estimated time: 45-60 minutes on T4 GPU

Using cuda device
Logging to ./tensorboard_logs/DQN_1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    exploration_rate | 0.783    |
| time/               |          |
|    episodes         | 100      |
|    fps              | 251      |
|    time_elapsed     | 87       |
|    total_timesteps  | 21934    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0146   |
|    n_updates        | 2983     |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.568    |
| time/               |          |
|    episodes         | 200      |
|    fps              | 225      |
|    time_elapsed     | 193      |
|    total_timesteps  | 43603    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0106   |
|    n_updates        | 8400     |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rat

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Mean Reward: -11.90 +/- 3.11
Random baseline: -21.00
Improvement: 9.10


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ TRAINING COMPLETE!
The model has been downloaded as dqn_model.zip
Place it in your D:/formative3_group13/ folder
Run: python play.py
